# Num. Epoch = 5

## Px - scaling 0

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Px_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 31818.24it/s]


Wasserstein-2 ε statistics over 44850 pairs:
----------------------------------------
max ε   : 6.955209
mean ε  : 4.299053
median ε: 3.764169
std ε   : 2.450638
min ε   : 0.188467


In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Px_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD pairs: 100%|██████████| 44850/44850 [00:01<00:00, 35201.45it/s]


Jensen–Shannon ε statistics over 44850 pairs:
--------------------------------------------------
max ε   : 1.421749
mean ε  : 0.953270
median ε: 1.081005
std ε   : 0.415024
min ε   : 0.105153


## Px - scaling 1

In [1]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Px_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 29996.13it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 35.415227
mean ε  : 23.581088
median ε: 34.145474
std ε   : 15.462385
min ε   : 0.136387


In [2]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Px_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 18563.66it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 2.829321
mean ε  : 1.510221
median ε: 1.550419
std ε   : 0.749826
min ε   : 0.085628


## Px - scaling 2

In [6]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Px_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 39514.66it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 35.822243
mean ε  : 24.068044
median ε: 34.150411
std ε   : 14.868181
min ε   : 0.193859


In [5]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Px_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:01<00:00, 25967.93it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 3.035017
mean ε  : 1.497934
median ε: 1.197654
std ε   : 0.758329
min ε   : 0.091892


.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.



# Num. Epoch = 5


## Py - scaling 0

In [7]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Py_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:00<00:00, 52884.43it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 37.701842
mean ε  : 23.186651
median ε: 25.558693
std ε   : 13.872279
min ε   : 0.316225


In [8]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Py_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:01<00:00, 30143.54it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 40.764161
mean ε  : 25.521711
median ε: 28.748024
std ε   : 15.765937
min ε   : 0.000162


## Py - scaling 1


In [9]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Py_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:00<00:00, 55696.94it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 46.672682
mean ε  : 30.974865
median ε: 32.258174
std ε   : 14.881347
min ε   : 0.303787


In [10]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Py_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:01<00:00, 30894.94it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 49.630881
mean ε  : 33.484586
median ε: 35.101934
std ε   : 16.476022
min ε   : 0.001341


## Py - scaling 2

In [11]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Py_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:00<00:00, 51342.05it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 45.827626
mean ε  : 33.786205
median ε: 42.382476
std ε   : 14.140358
min ε   : 0.197952


In [12]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Py_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:01<00:00, 30353.82it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 48.027314
mean ε  : 36.684887
median ε: 47.740678
std ε   : 15.552769
min ε   : 0.000169


.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.

# Num. Epoch = 5

## P(X|Y) - scaling 0

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Pxy_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Pxy_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

## P(X|Y) - scaling 1

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Pxy_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Pxy_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

## P(X|Y) - scaling 2

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Pxy_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Pxy_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.

# Num. Epoch = 5

## P(Y|X) - scaling 0

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Pyx_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Pyx_scaling0_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

## P(Y|X) - scaling 1

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Pyx_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Pyx_scaling1_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

## P(Y|X) - scaling 2

In [ ]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Pyx_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

In [ ]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("mnist_epsilon_umap/temp_Pyx_scaling2_ep5/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")